# OpenAQ - Pollution Map

### Imports

In [2]:
%load_ext autoreload
%autoreload 2

import time

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from openaq_anomaly_prediction.core.xgboost import CustomXGBoostModel

from openaq_anomaly_prediction.config import Configuration as cfg
from openaq_anomaly_prediction.load.gcp import bq
from openaq_anomaly_prediction.core.seoul_map import SeoulMap

### BigQuery

In [5]:
# ================================================================
# BIG QUERY REQUESTS:

# BigQuery
query = """
    SELECT
        *
    FROM `openaq-anomaly-prediction.dev_serving.ml_seoul__xgb_features`
    -- FROM `openaq-anomaly-prediction.dev_intermediate.int_seoul__measurements_segments`
    WHERE datetimeto_utc <= '2025-12-31'
    -- WHERE datetimeto_utc >= '2025-11-01'
    --   AND datetimeto_utc <= '2025-12-31'
    --  AND sensor_name = 'pm25 µg/m³'
    --    AND segment_total_hours >= 504
    ORDER BY location_id, datetimeto_utc ASC
"""
query_parameters = [
    # ("test_locations_ids", "STRING", TEST_LOCATIONS)
]
bigquery_df = bq.query(query, query_parameters=query_parameters, dry_run=False, debug=True)

# Results
if bigquery_df is not None:
    display(bigquery_df.head(1))
    # display(bigquery_df.info())

11:25:10      DEBUG |  > [BigQuery]: Request finished in 27.47s  ->  1,353,862 rows, 1593.23 MB


,location_id,datetimeto_utc,concurrent_sensors,segment_id,segment_start,segment_end,segment_perfect_hours,segment_total_hours,segment_perfect_density,segment_tier,...,feat_weather__moisture_saturation_index,feat_weather__dew_point_depression,feat_weather__photolytic_flux_index,feat_weather__wind_burst_ratio_index_10m,feat_weather__o3_potential_lag168,feat_weather__sulfate_potential_lag168,feat_weather__nitrate_potential_lag168,feat_weather__lag1_precipitation,feat_weather__delta1_pressure_msl,feat_weather__delta3_temperature_2m
0,2622586,2024-04-30 15:00:00+00:00,6,2622586_S1,2024-04-30 15:00:00+00:00,2024-05-15 15:00:00+00:00,357,361,0.98892,gold,...,2.0,4.5,0.0,1.601562,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
import gc
import os


output_path = os.path.join(cfg.DATA_EXPORT_PATH, "xgb_features_3months.parquet")

try:
    bigquery_df.to_parquet(output_path, index=False, compression="snappy")
    export_df = bigquery_df.copy()

except Exception as e:
    print(f"Error saving parquet file: {e}")
    print("Loading parquet file from disk instead...")
    export_df = pd.read_parquet(output_path)


# # ---------------------------------------------------------------------
# # Unload the dataframes
# export_df = None
# del export_df
# unreachable = gc.collect()

# export_df.columns.tolist()

In [ ]:
# For a given date (e.g., 2025-10-15), filter the dataframe to include only rows from that date
date_filter = '2025-12-11'  # PM25 peak at lunch
# date_filter = '2025-12-20'  # PM25 peak at 15h
date_filter = '2025-12-09'  # O3 peak (MOVING WAVES!)

daily_df = export_df[export_df['oaq_datetimeto_local'].dt.date == pd.to_datetime(date_filter).date()].copy()

# Interpolate missing pollutant values
daily_df = CustomXGBoostModel().interpolate_pollutants(daily_df, debug=True)

# Add the 'hour_of_day' column
daily_df['local_hour'] = daily_df['oaq_datetimeto_local'].dt.hour
daily_df.loc[:, "oaq_datetimeto_local_str"] = daily_df["oaq_datetimeto_local"].dt.strftime("%Y-%m-%d %H:%M")


INTERPOLATING 6 POLLUTANTS
----------------------------------------------------------------
BEFORE: 259 total NaN values
    -> Adding columns       : ['pm25_ugm3_is_imputed', 'pm10_ugm3_is_imputed', 'no2_ppm_is_imputed', 'o3_ppm_is_imputed', 'so2_ppm_is_imputed', 'co_ppm_is_imputed', 'any_imputed']
    -> Grouped by           : ['location_id', 'segment_id']
    -> Linear interpolation : up to 4 hours
    -> Forward fill         : up to 4 hours
    -> Backward fill        : up to 4 hours
AFTER: 0 total NaN values


### Maps

In [62]:
seoul_map = SeoulMap()

pollutant_col = "pm25_ugm3"
pollutant_col = "o3_ppm"

smoothed_df = seoul_map.smooth_dataframe_for_animation(
    daily_df,
    date_col='oaq_datetimeto_local',
    location_col='location_id',
    lat_col='oaq_latitude',
    lon_col='oaq_longitude',
    value_cols=[pollutant_col],
    freq='5min' # 15 minutes = 4x more frames. Use '5min' for ultra-smooth.
)
        
fig = seoul_map.scatter_map(
    smoothed_df,
    lat_column="oaq_latitude",
    lon_column="oaq_longitude",
    title="SEOUL - Scatter Map",
    label="PM 2.5 ug/m³",
    animation_frame = "animation_step",
    animation_group = "location_id",
    color_column=pollutant_col,
    size_column=pollutant_col,
    size_max=72,
    exponential=True,
    # coloraxes_min=0,
    coloraxes_min=smoothed_df[pollutant_col].min(),
    coloraxes_max=smoothed_df[pollutant_col].max(),
    map_style="basic",
    # map_style="satellite",
    hover_name="location_id",
    hover_data={
        "location_id": True,
        # "location_municipality": True,
        # "nan_pct": ":.2%",
        "oaq_latitude": False,
        "oaq_longitude": False,
        pollutant_col: ":.2f",
        # "location_perfect_hours": True,
        # "location_type": True,
    },
    map_layers=True,
    map_layers_glowing_border=False,
    map_layers_region = "seoul"
)

fig.show()

In [217]:
# import requests

# # 1. URL for Nationwide Municipalities (SIG)
# url = "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-municipalities-2018-geo.json"

# print(f"Fetching from: {url} ...")
# response = requests.get(url)

# if response.status_code != 200:
#     print(f"❌ Error: Status Code {response.status_code}")
# else:
#     data = response.json()
#     count = len(data['features'])
#     print(f"✅ Success! Downloaded {count} regions.")
    
#     # 2. Inspect the first item to see the REAL property keys
#     first_prop = data['features'][0]['properties']
#     print("\n--- Properties of the first region ---")
#     print(first_prop) 
#     # This will tell us if it is 'name_en', 'name_eng', or something else.
    
#     # 3. Test filter for a neighbor
#     test_neighbor = "Goyang"
#     found = False
#     for f in data['features']:
#         p = f['properties']
#         # Check all values in properties to see where "Goyang" is hiding
#         if test_neighbor in str(p.values()):
#             print(f"\n✅ Found Neighbor Test: {p}")
#             found = True
#             break
    
#     if not found:
#         print(f"\n❌ Could not find '{test_neighbor}' in any property.")

In [159]:
# seoul_map = SeoulMap()

# daily_df.loc[:, "datetimeto_utc_str"] = daily_df["datetimeto_utc"].dt.strftime("%Y-%m-%d %H:%M")

# hours = []
# for hour in sorted(daily_df["hour_of_day"].unique()):
#     # count_locations = daily_df[daily_df["hour_of_day"] == hour]["location_id"].nunique()
#     # print(f"Hour {hour:02d}: {count_locations} unique locations")
#     # Filter by hour_of_day (e.g., 14 for 2 PM)
#     hour_filter = 18
#     hour_df = daily_df[daily_df['hour_of_day'] == hour]
#     hours.append(hour_df)
    
    
# fig = seoul_map.scatter_map(
#     daily_df,
#     lat_column="oaq_latitude",
#     lon_column="oaq_longitude",
#     title="SEOUL - Scatter Map",
#     animation_frame = "datetimeto_utc_str",
#     animation_group = "location_id",
#     color="pm25_ugm3",
#     size_column="pm25_ugm3",
#     size_max=96,
#     exponential=True,
#     # title_config={"upper": False},
#     # colorbar_config={"option": "horizontal"},
#     # coloraxes_config={
#     #     "point_coloraxis": KOREAN_FLAG_CS,
#     #     "border_coloraxis": WHITE_TRANSPARENT_CS,
#     #     "cmin": 0,
#     #     "cmax": 2,
#     # },
#     coloraxes_min=daily_df["pm25_ugm3"].min(),
#     coloraxes_max=daily_df["pm25_ugm3"].max(),
#     map_style="basic",
#     hover_name="location_id",
#     hover_data={
#         "location_id": True,
#         # "location_municipality": True,
#         # "nan_count": True,
#         # "nan_pct": ":.2%",
#         "oaq_latitude": False,
#         "oaq_longitude": False,
#         # "location_perfect_hours": True,
#         # "location_type": True,
#     },
# )

# # Customize points
# # fig = seoul_map.customize_points(
# #     fig,
# #     daily_df,
# #     lat="oaq_latitude",
# #     lon="oaq_longitude",
# #     color="pm25_ugm3",
# #     size="pm25_ugm3",
# #     size_max=96,
# #     exponential=True,
# #     hover_name="location_id",
# #     hover_data={
# #         "location_id": True,
# #         # "location_municipality": True,
# #         # "nan_count": True,
# #         # "nan_pct": ":.2%",
# #         "oaq_latitude": False,
# #         "oaq_longitude": False,
# #         # "location_perfect_hours": True,
# #         # "location_type": True,
# #     },
# #     keep_animation=True,
# # )

# fig.show()


# # fig.update_layout(
# #     # annotations=seoul_map.customize_title("SEOUL - Scatter Map", upper=False),
# #     # # Color scales
# #     # **seoul_map.customize_coloraxes(
# #     #     KOREAN_FLAG_CS,
# #     #     WHITE_TRANSPARENT_CS,
# #     #     # cmax=perfect_locations_df["perfect_hour_measurements"].max()
# #     #     cmax=2,
# #     # ),
# #     # coloraxis_colorbar=customize_colorbar(option="vertical", x=0.91, y=0.82),
# #     # Legend
# #     # coloraxis_colorbar=seoul_map.customize_colorbar(option="horizontal"),
# #     # Map overlays
# #     # map_layers=seoul_map.customize_map_layers(
# #     #     seoul_map.seoul_geojson,
# #     #     seoul_map.seoul_geojson_districts,
# #     #     border_color="rgba(255, 255, 255, 1)",
# #     #     border_width=4,
# #     #     sub_border_color="rgba(160, 160, 160, 0.4)",
# #     #     sub_border_width=2,
# #     #     sub_dashes=[3, 2],
# #     #     dimmer_color="rgba(80, 80, 80, 0.4)",
# #     #     glowing_border=False,
# #     # ),
# # )